# PedRL — Pedagogical RL on GSM8K (Colab)

Implementation of [Pedagogical RL](https://noahziems.com/pedagogical-rl): train a **privileged teacher** (it sees the gold answer) with GRPO to maximize `correctness × G_spike`, where `G_spike` is a spike-aware learnability score under the **frozen student** — then distill the teacher into the student with **surprisal-gated** cross-entropy.

Repo: [dannyyoon0303/PedRL](https://github.com/dannyyoon0303/PedRL)

**Runtime:** use a GPU runtime (`Runtime → Change runtime type → T4/L4/A100`).

| preset | what | time on T4 | time on A100 |
|---|---|---|---|
| `smoke` | pipeline check (16 problems, 6 GRPO steps) | ~10–15 min | ~4 min |
| `poc` | real run (256 problems, 120 GRPO steps) | ~2–3 h | ~40 min |


In [ ]:
!nvidia-smi

## 1. Install dependencies

In [ ]:
%pip install -q -U "trl>=0.17.0" "peft>=0.14.0" "datasets>=2.19.0" "accelerate>=0.34.0" "transformers>=4.49.0"


## 2. Get the code

In [ ]:
import os
if not os.path.exists("PedRL"):
    !git clone https://github.com/dannyyoon0303/PedRL.git
%cd PedRL
!ls

## 3. Smoke test (~10 min on T4)

Runs the full pipeline on a tiny slice: baseline eval → teacher GRPO → corpus → gated assimilation → student eval. Numbers will be noisy — this only verifies everything runs.

In [ ]:
!python run.py all --preset smoke

## 4. Proof of concept

The real run. Each stage runs in its own process, so you can also re-run stages individually (e.g. after a Colab disconnect — completed stages leave artifacts in `outputs/`).

Watch the `[reward] acc=... G=... r_ped=...` lines during the teacher stage: accuracy should stay high (the teacher knows the answer) while `G` — learnability under the frozen student — climbs. Every reward batch is also logged to `outputs/reward_log_teacher.jsonl`, and the teacher adapter is checkpointed every 30 steps for the learning-curve analysis below.

In [ ]:
# fresh outputs for the PoC (the smoke test wrote to outputs/ too)
!rm -rf outputs
!python run.py all --preset poc

## 5. Results — pass@1

In [ ]:
import json, glob

for path in sorted(glob.glob("outputs/eval_*.json")):
    with open(path) as f:
        r = json.load(f)
    print(f"{r['tag']:>22}: pass@1 = {r['accuracy']:.3f}  (n={r['n']})")

## 5a. Does surprisal decrease as the teacher trains?

The core claim of Pedagogical RL: GRPO on `correctness × G_spike` should push the teacher toward trajectories the student finds *plausible*. If it's working, the surprisal gaps `d_t` (measured under the frozen student) fall and `G_spike` rises over training, while correctness stays high — the teacher already knows the answer, it's learning *how to say it*.

In [ ]:
# shared plot style (light surface; series colors: fixed categorical order)
import matplotlib.pyplot as plt

C1, C2 = "#2a78d6", "#1baf7a"      # series 1 (blue), series 2 (aqua)
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e0"

def style_ax(ax, xlabel="", ylabel="", title=""):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(GRID)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK2, labelsize=9)
    ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    ax.set_ylabel(ylabel, color=INK2, fontsize=10)
    ax.set_title(title, color=INK, fontsize=11, loc="left", pad=10)

def smooth(xs, w=9):
    if len(xs) < w: return xs
    out = []
    for i in range(len(xs)):
        lo = max(0, i - w // 2); hi = min(len(xs), i + w // 2 + 1)
        out.append(sum(xs[lo:hi]) / (hi - lo))
    return out


In [ ]:
import json

log = [json.loads(l) for l in open("outputs/reward_log_teacher.jsonl")]
x = [r["rollouts"] for r in log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
fig.patch.set_facecolor("white")

# panel 1: surprisal gaps under the frozen student (nats)
for key, color, label in [("mean_gap", C1, "mean gap"), ("mean_max_gap", C2, "max gap")]:
    y = [r[key] for r in log]
    ax1.plot(x, y, color=color, linewidth=1, alpha=0.25)
    ax1.plot(x, smooth(y), color=color, linewidth=2, label=label)
    ax1.annotate(label, (x[-1], smooth(y)[-1]), xytext=(6, 0),
                 textcoords="offset points", color=INK2, fontsize=9, va="center")
style_ax(ax1, "teacher rollouts", "surprisal gap d_t (nats)",
         "Student surprisal of teacher completions")
ax1.legend(frameon=False, fontsize=9, labelcolor=INK2)
ax1.margins(x=0.12)

# panel 2: learnability score and teacher correctness (both 0–1)
for key, color, label in [("mean_g", C1, "G_spike"), ("acc", C2, "teacher acc")]:
    y = [r[key] for r in log]
    ax2.plot(x, y, color=color, linewidth=1, alpha=0.25)
    ax2.plot(x, smooth(y), color=color, linewidth=2, label=label)
    ax2.annotate(label, (x[-1], smooth(y)[-1]), xytext=(6, 0),
                 textcoords="offset points", color=INK2, fontsize=9, va="center")
style_ax(ax2, "teacher rollouts", "score (0–1)", "Learnability and correctness")
ax2.set_ylim(0, 1.02)
ax2.legend(frameon=False, fontsize=9, labelcolor=INK2)
ax2.margins(x=0.12)

plt.tight_layout()
plt.show()

In [ ]:
# peek at a teacher demonstration the student found most learnable
import json
rows = [json.loads(l) for l in open("outputs/distill_corpus.jsonl")]
rows.sort(key=lambda r: -r["g_spike"])
r = rows[0]
print(f"G_spike={r['g_spike']:.3f}  gold={r['answer']}\n")
print(r["completion_text"][:1500])

## 5b. How fast does the eval score improve? (sample efficiency)

The blog's headline claim is that Pedagogical RL needs **fewer rollouts** than vanilla RL. To test that at PoC scale we compare, at matched rollout budgets:

- **PedRL curve** — for each teacher checkpoint (steps 0/30/60/90/120), build a corpus, assimilate a student, evaluate it. Step 0 = the *untrained* privileged teacher, i.e. plain rejection-sampling distillation.
- **Vanilla GRPO curve** — standard correctness-only GRPO on the student (no privileged info, same rollout budget), evaluated at the same checkpoints.

⏱️ This is the expensive section (~1.5–2.5 h extra on a T4; each PedRL point costs a corpus + assimilation + eval). Fewer points: `--set checkpoint_every=60` before the training stages, or shrink `curve_n_distill` / `curve_n_eval`.

In [ ]:
# vanilla-GRPO baseline at the same rollout budget as the teacher (writes checkpoints too)
!python run.py baseline-rl --preset poc

In [ ]:
# learning curves: eval accuracy vs rollouts (writes outputs/curve_*.json)
!python run.py curve-baseline --preset poc
!python run.py curve-pedrl --preset poc

In [ ]:
import json

fig, ax = plt.subplots(figsize=(7, 4.2))
fig.patch.set_facecolor("white")

with open("outputs/eval_base.json") as f:
    base_acc = json.load(f)["accuracy"]
ax.axhline(base_acc, color=INK2, linewidth=1.2, linestyle=(0, (4, 4)))
ax.annotate("base model", (0, base_acc), xytext=(4, 5),
            textcoords="offset points", color=INK2, fontsize=9)

for name, color, label in [("pedrl", C1, "PedRL (teacher rollouts)"),
                           ("baseline", C2, "vanilla GRPO")]:
    with open(f"outputs/curve_{name}.json") as f:
        pts = json.load(f)["points"]
    xs = [p["rollouts"] for p in pts]
    ys = [p["accuracy"] for p in pts]
    ax.plot(xs, ys, color=color, linewidth=2, marker="o", markersize=6, label=label)
    ax.annotate(label, (xs[-1], ys[-1]), xytext=(8, 0),
                textcoords="offset points", color=INK2, fontsize=9, va="center")

style_ax(ax, "training rollouts", "GSM8K pass@1",
         "Sample efficiency: PedRL vs vanilla GRPO")
ax.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="lower right")
ax.margins(x=0.18)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

print("\nNote: PedRL's x-axis counts TEACHER rollouts; assimilation adds no rollouts")
print("(it reuses the corpus). Curve points use a reduced eval set (curve_n_eval),")
print("so they are noisier than the headline pass@1 numbers above.")

## 6. Optional: ablation & stage 3

- **No-gating ablation** — is the surprisal gate doing work, or is this just rejection-sampling SFT?
- **Stage 3** — continue with *standard* GRPO on the assimilated student (correctness-only reward, no privileged info).

In [ ]:
# ablation: plain SFT on the same corpus (writes eval_student_sft_ablation.json)
!python run.py assimilate --preset poc --no-gating
!python run.py eval-student --preset poc --no-gating

In [ ]:
# optional stage 3: standard GRPO on the student (writes eval_student_rl.json)
!python run.py student-rl --preset poc

---
References: [Pedagogical RL blog](https://noahziems.com/pedagogical-rl) · [OPSD](https://github.com/siyan-zhao/OPSD) · [GRPO](https://arxiv.org/abs/2402.03300)